# Shell Executor

本文档演示如何在 Runtime 容器中执行 Shell 命令。 Nexus 提供了两种方式：
1. **无状态命令执行**：使用 `run_command` 执行单次命令
2. **有状态终端会话**：使用 Terminal Session 系列工具管理持久化的交互式终端

## 初始化环境

首先导入必要的模块并配置日志。

In [1]:
from nexus.utils.log import setup_logging
from nexus.runtime import DockerRuntimeProvider

setup_logging()

## 创建并获取 Runtime 实例

创建 Docker Runtime Provider 并获取一个 Ubuntu 24.04 的运行时实例。

In [2]:
rp = DockerRuntimeProvider()
runtime = await rp.acquire(name='main', image='ubuntu:24.04')

2026-01-13 02:42:11 | INFO | nexus.runtime.docker | Image nexus-runtime:0.8.6 already exists, skipping build
2026-01-13 02:42:11 | INFO | nexus.runtime.docker | Runtime container nexus-runtime-0-8-6 already exists
2026-01-13 02:42:11 | INFO | nexus.runtime.docker | Starting container nexus-user-main...
2026-01-13 02:42:15 | INFO | nexus.runtime.docker | Runtime main ready at http://172.17.0.1:33157


## 方式一：无状态命令执行（run_command）

**方法签名**

```python
async def run_command(
    command: str,
    *,
    cwd: str | None = None,
    timeout: float = 180,
    stdin: str | None = None,
    extra_env: dict[str, str] | None = None,
) -> RunCommandResponse
```

**参数说明**

- `command` (`str`)：要执行的 Shell 命令
- `cwd` (`str | None`)：命令的工作目录（绝对路径），默认为 `/home/user`
- `timeout` (`float`)：命令超时时间（秒），默认 180 秒
- `stdin` (`str | None`)：传递给命令标准输入的字符串
- `extra_env` (`dict[str, str] | None`)：额外的环境变量（会与容器环境合并）

**返回值**

`RunCommandResponse` 包含以下字段：
- `status` (`CommandRunStatus`)：执行状态，可能值为 `Finished`、`TimeLimitExceeded`、`Error`
- `stdout` (`str`)：标准输出内容
- `stderr` (`str`)：标准错误内容
- `return_code` (`int | None`)：进程退出码（超时时为 `None`）
- `execution_time` (`float`)：命令执行时长（秒）

**实现位置**

- Executor 实现：`nexus/extensions/executors/command.py`
- Mixin 方法：`nexus/runtime/executor_mixin.py`

### 基本用法示例

In [3]:
# 执行简单命令
result = await runtime.run_command('ls -la /home')
print(f"Status: {result.status}")
print(f"Return Code: {result.return_code}")
print(f"Output:\n{result.stdout}")

2026-01-13 02:42:15 | INFO | nexus.core.registry | Imported 'nexus.extensions.agents.demo.tools' with 2 registrations:
2026-01-13 02:42:15 | INFO | nexus.core.registry |   - [tool] demo/think, demo/bash
2026-01-13 02:42:15 | INFO | nexus.core.registry | Imported 'nexus.extensions.agents.swe_agent.tools' with 32 registrations:
2026-01-13 02:42:15 | INFO | nexus.core.registry |   - [executor] create_browser_session, destroy_browser_session, browser_goto, browser_go_back, browser_go_forward, browser_click, browser_fill, browser_scroll, browser_press_key, browser_hover, browse [... (truncated log)] l_session, run_command_in_terminal_session, start_command_in_terminal_session, query_terminal_command_status, send_keys_to_terminal_session, capture_terminal_pane, terminate_terminal_session_processes
2026-01-13 02:42:15 | INFO | nexus.core.registry |   - [tool] swe_agent/bash, swe_agent/str_replace_editor, swe_agent/review_submit
2026-01-13 02:42:16 | INFO | nexus.core.registry | Imported 'nexu

In [4]:
# 指定工作目录和环境变量
result = await runtime.run_command(
    'echo "MY_VAR=$MY_VAR"',
    cwd='/tmp',
    extra_env={'MY_VAR': 'hello'}
)
print(result.stdout)

MY_VAR=hello



## 方式二：有状态终端会话（Terminal Session）

Terminal Session 基于 tmux 提供持久化的交互式终端环境，适合需要保持状态的场景（如启动服务器、交互式程序等）。

### 1. create_terminal_session - 创建终端会话

**参数：**
- `session_id` (`str | None`)：会话 ID，默认自动生成 UUID
- `height` (`int`)：终端高度（行数），默认 40
- `width` (`int`)：终端宽度（字符数），默认 120

**返回值：**
- `session_id` (`str`)：创建的会话 ID

In [5]:
# 创建终端会话
session = await runtime.create_terminal_session(session_id='demo-session')
print(f"Created session: {session.session_id}")

Created session: demo-session


### 2. run_command_in_terminal_session - 同步执行命令

在终端会话中执行命令并等待完成。

**参数：**
- `session_id` (`str`)：会话 ID
- `command` (`str`)：要执行的命令
- `timeout` (`float | None`)：超时时间（秒）

**返回值：**
`CommandInfo` 包含：
- `command_id` (`int`)：命令 ID
- `command` (`str`)：执行的命令
- `exit_code` (`int | None`)：退出码
- `start_time` / `end_time` (`float`)：开始/结束时间戳
- `output` (`str`)：输出内容

In [6]:
# 同步执行命令（等待完成）
result = await runtime.run_command_in_terminal_session(
    session_id='demo-session',
    command='echo "Hello from terminal"'
)
print(f"Exit code: {result.exit_code}")
print(f"Output: {result.output}")

Exit code: 0
Output: Hello from terminal



### 3. start_command_in_terminal_session - 异步启动命令

启动命令但不等待完成，适合长时间运行的程序。

**参数：**
- `session_id` (`str`)：会话 ID
- `command` (`str`)：要执行的命令

**返回值：**
`CommandInfo`（命令刚启动时的状态，`end_time` 为 `None`）

In [7]:
# 异步启动命令（不等待完成）
result = await runtime.start_command_in_terminal_session(
    session_id='demo-session',
    command='sleep 5 && echo "Done"'
)
print(f"Started command ID: {result.command_id}")
print(f"Command still running: {result.end_time is None}")

Started command ID: 2
Command still running: True


### 4. query_terminal_command_status - 查询命令状态

查询之前启动的命令的执行状态。

**参数：**
- `session_id` (`str`)：会话 ID
- `command_id` (`int`)：命令 ID

**返回值：**
`CommandInfo`（包含最新的执行状态和输出）

In [8]:
# 查询命令状态
import asyncio
await asyncio.sleep(6)  # 等待命令完成

status = await runtime.query_terminal_command_status(
    session_id='demo-session',
    command_id=result.command_id
)
print(f"Command finished: {status.end_time is not None}")
print(f"Output: {status.output}")

Command finished: True
Output: Done



### 5. send_keys_to_terminal_session - 发送按键

向终端会话发送按键序列（如 Ctrl-C、Enter 等）。

**参数：**
- `session_id` (`str`)：会话 ID
- `keys` (`str | list[str]`)：tmux 格式的按键（如 `'C-c'`、`'Enter'`、`'q'`）

**返回值：**
- `success` (`bool`)：是否成功发送

In [9]:
# 启动长时间运行的命令
await runtime.start_command_in_terminal_session(
    session_id='demo-session',
    command='tail -f /dev/null'
)

# 等待 1 秒
import asyncio
await asyncio.sleep(1)

# 发送 Ctrl-C 中断命令
result = await runtime.send_keys_to_terminal_session(
    session_id='demo-session',
    keys='C-c'
)
print(f"Keys sent: {result.success}")

Keys sent: True


### 6. capture_terminal_pane - 捕获终端内容

捕获终端会话的可见内容或完整历史。

**参数：**
- `session_id` (`str`)：会话 ID
- `capture_entire` (`bool`)：是否捕获全部历史，默认 `False`（仅捕获当前可见内容）

**返回值：**
- `content` (`str`)：捕获的终端内容

In [10]:
# 捕获终端当前内容
result = await runtime.capture_terminal_pane(
    session_id='demo-session',
    capture_entire=False
)
print(f"Terminal content:\n{result.content.strip()}")

Terminal content:
root@cf1ffe73e88f:/# nexus_capture demo-session
Capture session started: /tmp/nexus-terminal/demo-session
root@cf1ffe73e88f:/# echo "Hello from terminal"
Hello from terminal
root@cf1ffe73e88f:/# sleep 5 && echo "Done"
Done
root@cf1ffe73e88f:/# tail -f /dev/null
^C
root@cf1ffe73e88f:/#


### 7. terminate_terminal_session_processes - 终止会话进程

终止终端会话中运行的所有进程。

**参数：**
- `session_id` (`str`)：会话 ID

**返回值：**
- `success` (`bool`)：是否成功终止
- `error` (`str | None`)：错误信息（如果失败）

In [11]:
# 终止会话中的所有进程
result = await runtime.terminate_terminal_session_processes(
    session_id='demo-session'
)
print(f"Terminated: {result.success}")

Terminated: True


### 8. destroy_terminal_session - 销毁终端会话

销毁终端会话并清理相关资源。

**参数：**
- `session_id` (`str`)：会话 ID

**返回值：**
- `success` (`bool`)：是否成功销毁

In [12]:
# 销毁终端会话
result = await runtime.destroy_terminal_session(session_id='demo-session')
print(f"Session destroyed: {result.success}")

Session destroyed: True


## 两种方式对比

| 特性 | run_command | Terminal Session |
|------|-------------|------------------|
| 状态保持 | 无状态，每次独立执行 | 有状态，环境变量、工作目录等会保持 |
| 适用场景 | 单次命令、快速操作 | 交互式程序、长期运行的服务、需要多次命令交互 |
| 复杂度 | 简单，一次调用 | 需要管理会话生命周期 |
| 交互控制 | 不支持交互（仅 stdin） | 支持发送按键、捕获输出等交互操作 |
| 性能开销 | 较低 | 较高（需要维护 tmux 会话） |

### 实现位置

- Terminal Executor 实现：`nexus/extensions/executors/terminal.py`
- Mixin 方法：`nexus/runtime/executor_mixin.py`

## 清理资源

使用完毕后释放所有 Runtime 实例。

In [13]:
await rp.release_all()

2026-01-13 02:42:24 | INFO | nexus.runtime.docker | Stopping container nexus-user-main...
2026-01-13 02:42:24 | INFO | nexus.runtime.docker | Runtime main released
